In [ ]:
import csv
import time
import os

# =====================================================================
# GENERAZIONE DATASET MASSIVO E-COMMERCE: SCENARIO 1 (1 MILIONE)
# =====================================================================

def genera_csv_ecommerce_scenario1_massivo(nome_file, numero_totale_nodi=1000000):
    """
    Genera un dataset CSV massivo che modella lo Scenario 1 (E-commerce).
    Implementa una scrittura a blocchi (chunking) per ottimizzare l'uso 
    della memoria RAM durante la generazione del catalogo su larga scala.
    """
    print(f"Inizio generazione dataset massivo E-commerce: {nome_file}")
    print(f"Nodi totali (Prodotti del catalogo): {numero_totale_nodi}")
    
    inizio = time.time()
    
    # 1. Preparazione dell'ambiente di esportazione
    cartella_destinazione = os.path.dirname(nome_file)
    if cartella_destinazione and not os.path.exists(cartella_destinazione):
        os.makedirs(cartella_destinazione)
        
    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])
        
        # 2. Inizializzazione della logica a blocchi (Chunking)
        blocco_dati = []
        dimensione_blocco = 50000 
        
        # 3. Generazione topologica: i prodotti isolati (3 -> N) puntano ai Best-Seller (0, 1, 2)
        for prodotto in range(3, numero_totale_nodi):
            blocco_dati.append([prodotto, 0])
            blocco_dati.append([prodotto, 1])
            blocco_dati.append([prodotto, 2])
            
            # 4. Scrittura su disco e svuotamento del blocco dati per il rilascio della memoria
            if len(blocco_dati) >= dimensione_blocco * 3:
                writer.writerows(blocco_dati)
                blocco_dati.clear()
                
        # 5. Scrittura finale degli ultimi dati rimasti in memoria
        if blocco_dati:
            writer.writerows(blocco_dati)
            
    fine = time.time()
    print(f"Generazione completata con successo in {fine - inizio:.2f} secondi.")
    print(f"File salvato in: {nome_file}\n")


# =====================================================================
# ESECUZIONE SCRIPT
# =====================================================================
if __name__ == "__main__":
    N_NODI = 1000000
    NOME_FILE = '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario1_1MILIONE_cs2.csv'
    
    genera_csv_ecommerce_scenario1_massivo(NOME_FILE, N_NODI)


In [ ]:
import csv
import random
import time
import os

# =====================================================================
# GENERAZIONE DATASET MASSIVO E-COMMERCE: SCENARIO 2 (RETE MISTA E HUB)
# =====================================================================

def genera_csv_ecommerce_scenario2_massivo(nome_file, n_nodi=1000000):
    """
    Genera un dataset CSV massivo (1 milione di nodi) modellando un e-commerce misto:
    - Best-Seller (Nodi 0-2) in un bundle chiuso.
    - Prodotti isolati (Nodi 3 - 499.999) senza cross-selling, puntano ai top.
    - Prodotti correlati (Nodi 500.000 - 999.999) con raccomandazioni organiche.
    - Hub (Prodotti civetta/base) eletti algoritmicamente.
    Implementa scrittura a blocchi (chunking) per l'ottimizzazione della RAM.
    """
    print(f"Inizio generazione dataset massivo E-Commerce (Scenario 2): {nome_file}")
    inizio = time.time()

    # Fissiamo il seed per garantire la riproducibilità della topologia stocastica
    random.seed(42)

    # 1. Preparazione dell'ambiente di esportazione
    cartella_destinazione = os.path.dirname(nome_file)
    if cartella_destinazione and not os.path.exists(cartella_destinazione):
        os.makedirs(cartella_destinazione)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        blocco_dati = []
        dimensione_blocco = 100000 

        def scrivi_blocco():
            """Funzione helper per il salvataggio dei dati sul disco (Chunking)."""
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        # 2. Topologia Best-Seller: Bundle/Circolo chiuso tra i nodi 0, 1 e 2
        blocco_dati.extend([[0, 1], [1, 2], [2, 0]])

        # 3. Generazione Prodotti Isolati (Nodi 3 -> 499.999)
        print("Generazione dei Prodotti Isolati (nessun cross-selling) in corso...")
        for prodotto_isolato in range(3, 500000):
            blocco_dati.extend([[prodotto_isolato, 0], [prodotto_isolato, 1], [prodotto_isolato, 2]])
            
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # 4. Generazione Prodotti Correlati e Hub Strategici (Nodi 500.000 -> 999.999)
        print("Generazione dei Prodotti Correlati e Hub in corso...")
        start_organici = 500000
        end_organici = 999999

        for prodotto_correlato in range(start_organici, end_organici + 1):
            blocco_dati.extend([[prodotto_correlato, 0], [prodotto_correlato, 1], [prodotto_correlato, 2]])

            # Distribuzione disomogenea delle raccomandazioni (Hub vs Normali)
            is_hub = (prodotto_correlato % 500 == 0)
            num_raccomandazioni = 25 if is_hub else 2

            for _ in range(num_raccomandazioni):
                target_casuale = random.randint(start_organici, end_organici) 
                if target_casuale != prodotto_correlato: 
                    blocco_dati.append([prodotto_correlato, target_casuale])

            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # 5. Scrittura finale degli ultimi dati rimasti in memoria
        scrivi_blocco()

    fine = time.time()
    print(f"Generazione completata con successo in {fine - inizio:.2f} secondi.")
    print(f"File salvato in: {nome_file}\n")


# =====================================================================
# ESECUZIONE SCRIPT
# =====================================================================
if __name__ == "__main__":
    NOME_FILE = '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario2_1MILIONE_cs2.csv'
    genera_csv_ecommerce_scenario2_massivo(NOME_FILE)


⚙️ Inizio generazione dataset massivo E-Commerce (Scenario 2): ../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario2_1MILIONE_cs2.csv
📊 Generazione dei Prodotti Isolati (nessun cross-selling) in corso...
🌐 Generazione dei Prodotti Correlati e Hub in corso...
✅ Generazione completata con successo in 1.27 secondi!
Il file '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario2_1MILIONE_cs2.csv' è pronto.


In [ ]:
import csv
import random
import time
import os

# =====================================================================
# GENERAZIONE DATASET MASSIVO E-COMMERCE: SCENARIO 3 (ORGANICO / SMALL-WORLD)
# =====================================================================

def genera_csv_ecommerce_scenario3_massivo(nome_file, n_nodi=1000000):
    """
    Genera un dataset CSV massivo (1 milione di nodi) modellando un catalogo e-commerce 
    con topologia Small-World. Implementa Cluster merceologici, co-acquisti (Triadi), 
    bridging mirato verso prodotti civetta e preferential attachment asimmetrico.
    Utilizza il chunking per la gestione ottimizzata della RAM.
    """
    print(f"Inizio generazione dataset massivo E-commerce (Scenario 3 Organico): {nome_file}")
    inizio = time.time()
    
    # Fissiamo il seed per la riproducibilità stocastica
    random.seed(42)

    # 1. Preparazione dell'ambiente di esportazione
    cartella_destinazione = os.path.dirname(nome_file)
    if cartella_destinazione and not os.path.exists(cartella_destinazione):
        os.makedirs(cartella_destinazione)

    with open(nome_file, mode='w', newline='') as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(['Source', 'Target'])

        blocco_dati = []
        dimensione_blocco = 100000 

        def scrivi_blocco():
            """Funzione helper per il salvataggio dei dati sul disco (Chunking)."""
            nonlocal blocco_dati
            if blocco_dati:
                writer.writerows(blocco_dati)
                blocco_dati.clear()

        best_sellers = [0, 1, 2] 
        
        # 2. Inizializzazione dei Micro-Hubs (Cluster merceologici strategici)
        micro_hubs = set(range(1000, n_nodi, 1000)) 
        micro_hubs_list = list(micro_hubs)
        
        # 3. Scambio Bidirezionale (Best-Seller -> Micro-Hub strategici)
        print("Generazione raccomandazioni inverse (Best-Seller -> Micro-Hub strategici)...")
        for bs in best_sellers:
            hub_selezionati = random.sample(micro_hubs_list, 150)
            for hub in hub_selezionati:
                blocco_dati.append([bs, hub]) 

        # 4. Rete Catalogo e Dinamiche Cross-Selling (Organiche)
        print("Generazione cross-selling organico (Cluster, Co-acquisti e Bridging)...")
        for prodotto in range(3, n_nodi):
            
            # A. Preferential Attachment Asimmetrico (Tutti puntano ai top)
            if random.random() < 0.90: blocco_dati.append([prodotto, 0]) 
            if random.random() < 0.70: blocco_dati.append([prodotto, 1]) 
            if random.random() < 0.50: blocco_dati.append([prodotto, 2]) 
            
            # B. Attività Locale (Cluster Merceologici)
            if prodotto in micro_hubs:
                for j in range(1, 15):
                    if prodotto + j < n_nodi:
                        blocco_dati.append([prodotto, prodotto + j]) 
            else:
                if prodotto + 1 < n_nodi:
                    blocco_dati.append([prodotto, prodotto + 1]) 
                
            # C. Chiusure Triadiche (Co-acquisti / Bought Together)
            if prodotto % 3 == 0 and (prodotto - 2) >= 3:
                blocco_dati.append([prodotto, prodotto - 2])
                
            # D. Bridging mirato (Suggerimenti cross-categoria)
            if random.random() < 0.05:
                if random.random() < 0.80:
                    target_casuale = random.choice(micro_hubs_list)
                else:
                    target_casuale = random.randint(3, n_nodi - 1) 
                
                if target_casuale != prodotto: 
                    blocco_dati.append([prodotto, target_casuale]) 
            
            if len(blocco_dati) >= dimensione_blocco:
                scrivi_blocco()

        # 5. Scrittura finale degli ultimi dati rimasti in memoria
        scrivi_blocco()

    fine = time.time()
    print(f"Generazione Scenario Massivo E-commerce completata con successo in {fine - inizio:.2f} secondi.\n")


# =====================================================================
# ESECUZIONE SCRIPT
# =====================================================================
if __name__ == "__main__":
    NOME_FILE = '../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario3_1MILIONE_cs2.csv'
    genera_csv_ecommerce_scenario3_massivo(NOME_FILE)


⚙️ Inizio generazione dataset massivo E-commerce (Scenario 3 Organico): ../Caso di Studio 2/DataSet_CasoStudio2/Rete_1M/dataset_scenario3_1MILIONE_cs2.csv
🎯 Generazione raccomandazioni inverse (Best-Seller -> Micro-Hub strategici)...
🌐 Generazione cross-selling organico (Cluster, Co-acquisti e Bridging)...
✅ Generazione Scenario Massivo E-commerce completata con successo in 1.18 secondi!
